# アンサンブル



In [1]:
from PIL import Image
import os
import hashlib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import segmentation_models_pytorch as smp


# 設定

data_dir = "content/my_dataset"

unet_model_path = "segmentation_baseline_models/fasc_model.pth"
deeplab_model_path = "segmentation_baseline_models/DeepLabV3_fasc_model.pth"

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def load_fasc_binary_mask(mask_dir, file_name):
    """
    fasc mask を
    背景=0
    fascicle=1
    の二値配列に変換する
    """
    mask = Image.open(os.path.join(mask_dir, file_name))
    mask_array = np.array(mask)

    if mask.mode == "L":
        binary_mask = (mask_array == 255).astype(np.uint8)
    else:
        raise ValueError(f"未対応のmodeです: {mask.mode}")

    return binary_mask


def calculate_sha256(file_path):
    """
    ファイルのSHA-256を計算する
    """
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


def resize_with_padding(image, mask, target_size=(768, 512)):
    """
    image と mask の縦横比を維持したまま縮小し、
    target_size まで余白を追加する
    """
    target_width, target_height = target_size
    original_width, original_height = image.size

    scale = min(target_width / original_width, target_height / original_height)

    new_width = int(original_width * scale)
    new_height = int(original_height * scale)

    image = image.resize((new_width, new_height), resample=Image.Resampling.BILINEAR)
    mask = mask.resize((new_width, new_height), resample=Image.Resampling.NEAREST)

    left = (target_width - new_width) // 2
    top = (target_height - new_height) // 2

    image_canvas = Image.new(image.mode, target_size, 0)
    mask_canvas = Image.new("L", target_size, 0)

    image_canvas.paste(image, (left, top))
    mask_canvas.paste(mask, (left, top))

    return image_canvas, mask_canvas


class SegmentationDataset(Dataset):

    def __init__(self, image_dir, mask_dir, mask_loader):

        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.mask_loader = mask_loader

        image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith(
                (".tif", ".tiff", ".png", ".jpg", ".jpeg")
            )
        ])

        #image + mask が完全一致する重複を除去
        unique_image_files = []
        seen_pairs = set()

        for file_name in image_files:

            image_path = os.path.join(image_dir, file_name)
            mask_path = os.path.join(mask_dir, file_name)

            image_hash = calculate_sha256(image_path)
            mask_hash = calculate_sha256(mask_path)

            pair_hash = (image_hash, mask_hash)

            if pair_hash in seen_pairs:
                continue

            seen_pairs.add(pair_hash)
            unique_image_files.append(file_name)

        self.image_files = unique_image_files

        print("=============================================")
        print("Dataset duplicate removal")
        print("before:", len(image_files))
        print("after:", len(self.image_files))
        print("removed:", len(image_files) - len(self.image_files))
        print("=============================================")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):

        file_name = self.image_files[index]

        image_path = os.path.join(self.image_dir, file_name)
        image = Image.open(image_path).convert("RGB")
        mask = self.mask_loader(self.mask_dir, file_name)

        #maskとimageをサイズ揃える
        mask = Image.fromarray(mask)
        mask = mask.resize(image.size, resample=Image.Resampling.NEAREST)
        image, mask = resize_with_padding(image, mask, target_size=(768, 512))

        # image → numpy → Tensor
        image = np.array(image)
        image = torch.from_numpy(image).float() / 255.0
        image = image.permute(2, 0, 1)

        # mask → numpy → Tensor
        mask = np.array(mask)
        mask = torch.from_numpy(mask).float()
        mask = mask.unsqueeze(0)

        return image, mask


# U-Net

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)

        self.dec1 = DoubleConv(128, 64)

        # 出力
        self.output = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):

        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        enc3 = self.enc3(self.pool2(enc2))
        enc4 = self.enc4(self.pool3(enc3))

        bottleneck = self.bottleneck(
            self.pool4(enc4)
        )

        dec4 = self.up4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)
        dec3 = self.up3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)
        dec2 = self.up2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)
        dec1 = self.up1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)

        return self.output(dec1)


# U-Net 読み込み

unet_model = UNet()

unet_checkpoint = torch.load(unet_model_path, map_location=device)
unet_model.load_state_dict(unet_checkpoint["model_state_dict"])
unet_model = unet_model.to(device)
unet_model.eval()


# DeepLabV3+ 読み込み

deeplab_model = smp.DeepLabV3Plus(encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
)

deeplab_checkpoint = torch.load(deeplab_model_path, map_location=device)
deeplab_model.load_state_dict(deeplab_checkpoint["model_state_dict"])
deeplab_model = deeplab_model.to(device)
deeplab_model.eval()


print("=============================================")
print("U-Net model をロードしました")
print(
    "U-Net threshold:",
    unet_checkpoint["best_threshold"]
)
print("DeepLabV3+ model をロードしました")
print(
    "DeepLabV3+ threshold:",
    deeplab_checkpoint["best_threshold"]
)
print("=============================================")


# Validation Dataset

fasc_images_new_model_v1 = os.path.join(data_dir, "fasc_imgs_v1", "fasc_images_new_model_v1")
fasc_masks_new_model_v1 = os.path.join(data_dir, "fasc_masks_v1", "fasc_masks_new_model_v1")

dataset = SegmentationDataset(
    fasc_images_new_model_v1,
    fasc_masks_new_model_v1,
    load_fasc_binary_mask
)

train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False
)


# Ensemble ratio × threshold 自動探索

fasc_thresholds = [
    round(0.20 + i * 0.005, 3)
    for i in range(151)
]

unet_ensemble_ratios = [
    round(i * 0.01, 2)
    for i in range(101)
]

threshold_tensor = torch.tensor(fasc_thresholds, device=device).view(-1, 1, 1, 1, 1)

intersection_totals = torch.zeros(
    (len(unet_ensemble_ratios), len(fasc_thresholds)),
    dtype=torch.float64,
    device=device
)

pred_totals = torch.zeros(
    (len(unet_ensemble_ratios), len(fasc_thresholds)),
    dtype=torch.float64,
    device=device
)

true_total = 0


with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        unet_outputs = unet_model(images)
        deeplab_outputs = deeplab_model(images)

        unet_probs = torch.sigmoid(unet_outputs)
        deeplab_probs = torch.sigmoid(deeplab_outputs)

        masks_bool = masks > 0.5
        true_total += masks_bool.sum().item()

        for ratio_index, unet_ensemble_ratio in enumerate(unet_ensemble_ratios):

            deeplab_ensemble_ratio = 1.0 - unet_ensemble_ratio

            ensemble_probs = (unet_ensemble_ratio * unet_probs + deeplab_ensemble_ratio * deeplab_probs)
            pred_masks = (ensemble_probs.unsqueeze(0) >= threshold_tensor)
            intersection_totals[ratio_index] += (pred_masks & masks_bool.unsqueeze(0)).sum(dim=(1, 2, 3, 4))
            pred_totals[ratio_index] += pred_masks.sum(dim=(1, 2, 3, 4))

dice_scores = (2 * intersection_totals) / (pred_totals + true_total)
best_flat_index = torch.argmax(dice_scores).item()
best_ratio_index = (best_flat_index // len(fasc_thresholds))
best_threshold_index = (best_flat_index % len(fasc_thresholds))
best_unet_ensemble_ratio = unet_ensemble_ratios[best_ratio_index]
best_deeplab_ensemble_ratio = (1.0 - best_unet_ensemble_ratio)
best_threshold = fasc_thresholds[best_threshold_index]
best_validation_dice = dice_scores[best_ratio_index, best_threshold_index].item()


print()
print("=============================================")
print("ENSEMBLE GRID SEARCH BEST RESULT")
print("=============================================")
print("search combinations:",len(unet_ensemble_ratios) * len(fasc_thresholds))
print("U-Net weight:", best_unet_ensemble_ratio)
print("DeepLabV3+ weight:", best_deeplab_ensemble_ratio)
print("best threshold:", best_threshold)
print("best validation Dice:", best_validation_dice)
print("=============================================")


/home/nk/kengo/Muscle_Architecture_in_Ultrasound_Data/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti
U-Net model をロードしました
U-Net threshold: 0.46
DeepLabV3+ model をロードしました
DeepLabV3+ threshold: 0.54
Dataset duplicate removal
before: 2761
after: 2091
removed: 670

ENSEMBLE GRID SEARCH BEST RESULT
search combinations: 15251
U-Net weight: 0.5
DeepLabV3+ weight: 0.5
best threshold: 0.51
best validation Dice: 0.30998707469499176
